# Saudi Digital Concierge — OpenAI API Single-Agent

A clean API-based alternative to the Qwen3 baseline. Uses the **same local Saudi tourism knowledge base** (FAISS embeddings + SQLite hotels/events) and the same five test scenarios.

**Run cells from top to bottom in Google Colab.** No GPU is required for the LLM; the embedding model runs on CPU. An OpenAI API key with available credits is required.

**Research note:** This is a different model/backend from Qwen. Label its results **OpenAI Single-Agent**; do not mix them with Qwen baseline results without noting the model change.

In [1]:
# 1 — Install dependencies
%pip install -q "openai>=1.50.0" sentence-transformers faiss-cpu "pandas==2.2.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.7 MB/s eta 0:00:00


In [2]:
# 2 — Imports and runtime
import os, json, time, re, inspect, sqlite3, io, contextlib
import numpy as np
import pandas as pd
import faiss
from getpass import getpass
from openai import OpenAI

print("Pandas:", pd.__version__)
print("OpenAI API mode — GPU not required")

Pandas: 2.2.3
OpenAI API mode — GPU not required


In [3]:
import os
import subprocess

REPO_URL = "https://github.com/bedooralmareni/saudi-Digital-Concierge.git"
PROJECT_DIR = "/content/saudi-Digital-Concierge"

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, PROJECT_DIR],
        check=True
    )

VECTOR_DIR = f"{PROJECT_DIR}/data/final/vector_store"
DB_PATH = f"{PROJECT_DIR}/data/final/knowledge_base.sqlite"

required_files = [
    f"{VECTOR_DIR}/vectors.npy",
    f"{VECTOR_DIR}/vector_meta.csv",
    f"{VECTOR_DIR}/index_info.json",
    DB_PATH
]

for path in required_files:
    assert os.path.isfile(path), f"Missing file: {path}"
    print("OK:", path)

OK: /content/saudi-Digital-Concierge/data/final/vector_store/vectors.npy
OK: /content/saudi-Digital-Concierge/data/final/vector_store/vector_meta.csv
OK: /content/saudi-Digital-Concierge/data/final/vector_store/index_info.json
OK: /content/saudi-Digital-Concierge/data/final/knowledge_base.sqlite


In [4]:
import json
import numpy as np
import pandas as pd
import faiss

with open(f"{VECTOR_DIR}/index_info.json", encoding="utf-8") as f:
    info = json.load(f)

vectors = np.load(f"{VECTOR_DIR}/vectors.npy")
metadata = pd.read_csv(f"{VECTOR_DIR}/vector_meta.csv")

assert info["model"] == "intfloat/multilingual-e5-base"
assert info["normalized"] is True
assert len(vectors) == len(metadata)
assert vectors.shape[1] == 768

vectors = np.ascontiguousarray(vectors, dtype="float32")
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("FAISS vectors:", index.ntotal)
print("Dimensions:", index.d)
print("SQLite exists:", os.path.isfile(DB_PATH))

FAISS vectors: 13920
Dimensions: 768
SQLite exists: True


In [5]:
# 5 — Load embedding model (CPU; OpenAI API generates answers)
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"
)
print("Embedding model ready.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model ready.


In [6]:
# 6 — Configure OpenAI API securely
# Never paste your API key into a notebook cell or commit it to GitHub.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

api_key = api_key or os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = getpass("Enter OpenAI API key (hidden): ").strip()

if not api_key:
    raise ValueError("OPENAI_API_KEY is required.")

client = OpenAI(api_key=api_key)
OPENAI_MODEL = "gpt-4o-mini"  # Change to a model available to your API project.
print("OpenAI client ready. Model:", OPENAI_MODEL)

Enter OpenAI API key (hidden): ··········
OpenAI client ready. Model: gpt-4o-mini


In [7]:


import sqlite3
import numpy as np
import pandas as pd
import faiss


# Shared Arabic / English city normalization
CITY_ALIASES = {
    "الرياض": "Riyadh", "riyadh": "Riyadh",
    "جدة": "Jeddah", "جده": "Jeddah", "jeddah": "Jeddah",
    "أبها": "Abha", "ابها": "Abha", "abha": "Abha",
    "الدمام": "Dammam", "dammam": "Dammam",
    "الطائف": "Taif", "taif": "Taif",
    "المدينة": "Madinah", "المدينة المنورة": "Madinah", "madinah": "Madinah",
    "مكة": "Makkah", "مكه": "Makkah", "makkah": "Makkah",
    "العلا": "Al Ula", "al ula": "Al Ula",
}

def normalize_city(city):
    cleaned = str(city).strip()
    return CITY_ALIASES.get(cleaned.casefold(), cleaned)

# ============================================
# TOOL 1: SEMANTIC SEARCH
# Attractions, Restaurants, Tourist Places
# ============================================

def search_places(query, city, entity_type=None, top_k=5):
    city = normalize_city(city)
    type_mapping = {
        "attraction": "entertainment", "attractions": "entertainment",
        "tourist attraction": "entertainment", "tourist attractions": "entertainment",
        "restaurant": "place", "restaurants": "place",
        "cafe": "place", "cafes": "place",
    }
    if entity_type:
        mapped = [type_mapping.get(t.strip().casefold(), t.strip().casefold())
                  for t in str(entity_type).split(",") if t.strip()]
        entity_type = ",".join(dict.fromkeys(mapped))

    # ----------------------------------------
    # Filter by city
    # ----------------------------------------

    city_mask = (
        metadata["city"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        == city.strip().casefold()
    )

    mask = city_mask.copy()

    # ----------------------------------------
    # Support multiple entity types
    # Example: "attraction,restaurant"
    # ----------------------------------------

    if entity_type:

        types = [
            t.strip().casefold()
            for t in entity_type.split(",")
            if t.strip()
        ]

        type_mask = (
            metadata["entity_type"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.casefold()
            .isin(types)
        )

        mask &= type_mask

    # ----------------------------------------
    # Get matching vector indices
    # ----------------------------------------

    ids = np.flatnonzero(mask.to_numpy())

    if len(ids) == 0:
        return []

    # ----------------------------------------
    # Build temporary FAISS index
    # ----------------------------------------

    selected = np.ascontiguousarray(
        vectors[ids],
        dtype="float32"
    )

    local_index = faiss.IndexFlatIP(
        selected.shape[1]
    )

    local_index.add(selected)

    # ----------------------------------------
    # Encode user query
    # ----------------------------------------

    q = embedder.encode(
        ["query: " + query],
        normalize_embeddings=True
    ).astype("float32")

    # ----------------------------------------
    # Semantic search
    # ----------------------------------------

    scores, local_ids = local_index.search(
        q,
        min(len(ids), top_k * 5)
    )

    # ----------------------------------------
    # Remove duplicate entities
    # ----------------------------------------

    results = []
    seen = set()

    for score, local_id in zip(
        scores[0],
        local_ids[0]
    ):

        if local_id < 0:
            continue

        row = metadata.iloc[
            int(ids[local_id])
        ]

        key = row.get("canonical_id")

        if pd.isna(key):
            key = row.get(
                "entity_id",
                int(ids[local_id])
            )

        key = str(key)

        # Also deduplicate records that refer to the same named venue.
        name_key = str(row.get("text", "")).split(" — ")[0].strip().casefold()
        dedup_key = name_key or key
        if dedup_key in seen:
            continue

        seen.add(dedup_key)

        results.append({
            "score": round(float(score), 4),
            "city": str(row.get("city", "")),
            "type": str(row.get("entity_type", "")),
            "category": str(row.get("category", "")),
            "text": str(row.get("text", ""))
        })

        if len(results) >= top_k:
            break

    return results


# ============================================
# TOOL 2: EVENT SEARCH
# Filter events by city and travel dates
# ============================================

def search_events(
    city,
    start_date,
    end_date,
    limit=10
):

    city = normalize_city(city)
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    if start > end:
        raise ValueError(
            "Invalid date range"
        )

    with sqlite3.connect(DB_PATH) as conn:

        df = pd.read_sql_query(
            """
            SELECT
                event_id,
                name,
                city,
                start_date,
                end_date,
                source,
                source_url

            FROM events

            WHERE
                LOWER(TRIM(city)) = LOWER(TRIM(?))

                AND date(start_date) <= date(?)

                AND date(
                    COALESCE(end_date, start_date)
                ) >= date(?)

            ORDER BY start_date

            LIMIT ?
            """,
            conn,
            params=(
                city,
                end.strftime("%Y-%m-%d"),
                start.strftime("%Y-%m-%d"),
                limit
            )
        )

    return df.to_dict(
        orient="records"
    )


# ============================================
# TOOL 3: HOTEL SEARCH
# Filter by city, budget, rating, guests
# ============================================

def search_hotels(
    city,
    max_price_per_night=None,
    min_guest_rating=None,
    guests=None,
    limit=5
):

    city = normalize_city(city)
    conditions = [
        "LOWER(TRIM(city)) = LOWER(TRIM(?))"
    ]

    params = [city]

    # ----------------------------------------
    # Budget filter
    # ----------------------------------------

    if max_price_per_night is not None:

        conditions.append(
            "price_sar <= ?"
        )

        params.append(
            max_price_per_night
        )

    # ----------------------------------------
    # Guest rating filter
    # ----------------------------------------

    if min_guest_rating is not None:

        conditions.append(
            "guest_rating >= ?"
        )

        params.append(
            min_guest_rating
        )

    # ----------------------------------------
    # Guest capacity filter
    # ----------------------------------------

    if guests is not None:

        conditions.append(
            "max_persons >= ?"
        )

        params.append(
            guests
        )

    where_clause = " AND ".join(
        conditions
    )

    query = f"""
        SELECT
            hotel_id,
            name,
            city,
            price_sar,
            star_rating,
            guest_rating,
            max_persons,
            source,
            snapshot_date

        FROM hotels

        WHERE {where_clause}

        ORDER BY
            guest_rating DESC,
            price_sar ASC

        LIMIT ?
    """

    with sqlite3.connect(DB_PATH) as conn:

        df = pd.read_sql_query(
            query,
            conn,
            params=params + [limit]
        )

    return df.to_dict(
        orient="records"
    )


# ============================================
# READY
# ============================================

print("Knowledge Base Search Tools Ready!")
print("1. search_places")
print("2. search_events")
print("3. search_hotels")

Knowledge Base Search Tools Ready!
1. search_places
2. search_events
3. search_hotels


In [8]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": (
                "Search hotels in a Saudi city by maximum nightly price, "
                "minimum guest rating (out of 10), and guest capacity. "
                "Prices are historical snapshots, not live quotes."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "max_price_per_night": {"type": "number"},
                    "min_guest_rating": {"type": "number"},
                    "guests": {"type": "integer"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_places",
            "description": (
                "Search attractions, restaurants and other "
                "tourism entities using semantic search. "
                "Use the city and user's interests."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "city": {"type": "string"},
                    "entity_type": {"type": "string"}
                },
                "required": ["query", "city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_events",
            "description": (
                "Find events recorded in the knowledge base "
                "that overlap the requested travel dates. "
                "Results do not confirm live availability."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "start_date": {"type": "string"},
                    "end_date": {"type": "string"}
                },
                "required": ["city", "start_date", "end_date"]
            }
        }
    }
]

TOOL_FUNCTIONS = {
    "search_hotels": search_hotels,
    "search_places": search_places,
    "search_events": search_events
}

print("Available tools:", list(TOOL_FUNCTIONS))

Available tools: ['search_hotels', 'search_places', 'search_events']


In [9]:
import inspect

def execute_tool(name, arguments):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"Unknown tool: {name}"}

    if not isinstance(arguments, dict):
        return {"error": "Arguments must be an object"}

    func = TOOL_FUNCTIONS[name]
    signature = inspect.signature(func)

    try:
        signature.bind(**arguments)
        result = func(**arguments)

        return json.loads(json.dumps(
            result,
            ensure_ascii=False,
            default=str
        ))

    except Exception as e:
        return {"error": str(e)}

In [ ]:
# 10 — OpenAI single-agent tool-calling loop

SYSTEM_PROMPT = '''
You are Saudi Digital Concierge, a Saudi tourism assistant.
Use the provided search tools to answer the user's exact request.
Use only retrieved hotel, place, restaurant, and event records for named recommendations.
Never invent places, prices, ratings, hours, event availability, or costs.
For hotel-only requests, return only hotel recommendations, not an itinerary.
For itinerary requests, make a day-by-day plan when supported by evidence.
Respect the city, capacity, rating, budget, dates, exclusions, and strict inequalities.
"Under 300" means < 300; "at most 400" means <= 400.
The search_hotels tool's maximum price filter is inclusive; for a strict
"under" requirement, verify the returned prices and exclude equal-price hotels.
Only call a place a restaurant if its retrieved category/text confirms it.
Never treat a hotel, cinema, or entertainment venue as a restaurant.
Do not recommend restaurants with missing ratings when a minimum rating is required.
If a request is infeasible, explain which constraints cannot be met from the
available data and suggest specific, reasonable changes without violating them.
If a requested date range is check-in to check-out, calculate nights as
(end date - start date).days, not the number of calendar days.
Report only known costs, clearly label unknown activity/meal/transport costs,
and do not claim the entire trip is within budget unless supported.
Hotel prices are historical snapshots, not current quotes; mention the snapshot date.
Stored event dates are not proof that events are currently available.
Do not invent exact dates if the user did not provide them.
Do not claim that a booking was made.
Answer in the same language as the original user request.
For Arabic requests, write headings and explanatory text in Arabic;
preserve official names as returned by tools.
'''

def detect_user_language(text):
    return "Arabic" if len(re.findall(r"[\u0600-\u06FF]", text)) > len(re.findall(r"[A-Za-z]", text)) else "English"

def wrong_language(answer, expected):
    # Ignore Latin text inside official names imperfectly; focus on prose.
    arabic = len(re.findall(r"[\u0600-\u06FF]", answer))
    english = len(re.findall(r"[A-Za-z]", answer))
    return (arabic < english * 0.35) if expected == "Arabic" else (english < arabic * 0.35)

def run_agent(user_request, max_steps=10, max_retries=2):
    started = time.time()
    language = detect_user_language(user_request)
    lang_rule = (
        "اكتب الإجابة النهائية باللغة العربية، مع إبقاء أسماء الأماكن الرسمية كما هي."
        if language == "Arabic" else "Write the final answer in English."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT + "\n" + lang_rule},
        {"role": "user", "content": user_request},
    ]
    trace = []
    last_answer = ""
    retries = 0

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=TOOLS,
            # Require retrieval first; allow a final answer afterward.
            tool_choice="required" if step == 1 else "auto",
            temperature=0,
        )
        msg = response.choices[0].message
        calls = msg.tool_calls or []

        if not calls:
            answer = (msg.content or "").strip()
            last_answer = answer
            if (not answer or wrong_language(answer, language)) and retries < max_retries:
                retries += 1
                messages.append({"role": "assistant", "content": answer or "(empty)"})
                messages.append({
                    "role": "user",
                    "content": (
                        "أعد كتابة الإجابة كاملة بالعربية مع الحفاظ على الحقائق المسترجعة فقط."
                        if language == "Arabic"
                        else "Rewrite the full answer in English, using only retrieved facts."
                    )
                })
                continue
            # Search tools return lists on success (including no matches).
            # Error dictionaries do not count as successful retrieval.
            retrieval_succeeded = any(
                isinstance(item["result"], list)
                for item in trace
            )
            if not retrieval_succeeded:
                return {
                    "status": "retrieval_failed",
                    "answer": "KB retrieval failed; no grounded answer was produced.",
                    "trace": trace,
                    "elapsed_seconds": round(time.time() - started, 2),
                }

            status = "completed" if answer and not wrong_language(answer, language) else "incomplete"
            return {
                "status": status, "answer": answer, "trace": trace,
                "elapsed_seconds": round(time.time() - started, 2)
            }

        # Append assistant tool-call message in OpenAI's native format.
        messages.append(msg.model_dump(exclude_none=True))

        for call in calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or "{}")
                result = execute_tool(name, args)
            except Exception as exc:
                args = {}
                result = {"error": str(exc)}

            trace.append({
                "step": step, "tool": name, "arguments": args, "result": result
            })
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result, ensure_ascii=False, default=str)
            })

    return {
        "status": "max_steps_reached",
        "answer": last_answer or "Maximum tool-calling steps reached without a final answer.",
        "trace": trace,
        "elapsed_seconds": round(time.time() - started, 2)
    }

print("OpenAI single-agent ready.")

In [11]:

#  FIVE BASELINE TEST SCENARIOS


import time

SCENARIOS = {

    # ========================================
    # S1 — Simple Hotel Recommendation
    # ========================================

    "S1": """
أبغى فندق في الرياض لشخصين،
سعر الليلة ما يتجاوز 400 ريال،
وتقييم النزلاء يكون 8 من 10 أو أعلى.
اقترح لي أفضل 3 خيارات.
""",

    # ========================================
    # S2 — Personalized Travel Plan
    # ========================================

    "S2": """
Plan a 3-day trip to Jeddah for two people.
I enjoy entertainment and family-friendly activities.
Recommend a hotel under 300 SAR per night
with a guest rating of at least 8/10,
along with suitable places to visit.
""",

    # ========================================
    # S3 — Nature Trip with Constraints
    # ========================================

    "S3": """
خطط لي رحلة يومين إلى أبها لشخصين.
نحب الطبيعة والأنشطة الخارجية،
ونبغى سكن بسعر أقل من 350 ريال لليلة
وتقييمه 7 من 10 أو أعلى.
رتب لنا جدولًا مناسبًا،
ولا تقترح أماكن خارج أبها.
""",

    # ========================================
    # S4 — Multi-Constraint Planning
    # ========================================

    "S4": """
Plan a 3-day trip to Riyadh for two people
from October 15 to 17, 2026.

Our total budget is 1,500 SAR,
including accommodation and activities.

We want cultural attractions, entertainment,
and restaurants rated at least 4/5.

Avoid shopping malls,
and include only events that overlap
our travel dates.

Provide a daily itinerary and
an estimated cost breakdown.

Clearly identify any costs that are
unavailable in the dataset.
""",

    # ========================================
    # S5 — Conflicting Constraints
    # ========================================

    "S5": """
أبغى رحلة إلى جدة لمدة 3 أيام لشخصين،
بميزانية إجمالية 300 ريال تشمل السكن والأنشطة.

لازم يكون الفندق تقييمه 9 من 10 أو أعلى
وسعر الليلة أقل من 50 ريال.

وأبغى مطعمين تقييمهم 4.5 من 5 أو أعلى.

لا تتجاوز الميزانية أو تتجاهل أي شرط،
وإذا الطلب غير ممكن وضح لي السبب
واقترح تعديلات مناسبة.
"""
}




In [12]:
# 12 — Run all five scenarios and print only request + answer
ALL_RESULTS = {}

for scenario_id, user_request in SCENARIOS.items():
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            result = run_agent(user_request, max_steps=10, max_retries=2)
    except Exception as exc:
        result = {
            "status": "error", "answer": f"API/tool error: {exc}",
            "trace": [], "elapsed_seconds": 0
        }

    ALL_RESULTS[scenario_id] = {
        "request": user_request.strip(),
        "answer": result["answer"],
        "status": result["status"],
        "trace": result["trace"],
        "elapsed_seconds": result["elapsed_seconds"],
        "model": OPENAI_MODEL
    }
    print("\n" + "-" * 65)
    print(f"SCENARIO {scenario_id}")
    print("-" * 65)
    print("\nUSER REQUEST:\n")
    print(user_request.strip())
    print("\nOPENAI ANSWER:\n")
    print(result["answer"])

# Save detailed traces privately in Colab for evaluation.
with open("openai_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(ALL_RESULTS, f, ensure_ascii=False, indent=2, default=str)


-----------------------------------------------------------------
SCENARIO S1
-----------------------------------------------------------------

USER REQUEST:

أبغى فندق في الرياض لشخصين،
سعر الليلة ما يتجاوز 400 ريال،
وتقييم النزلاء يكون 8 من 10 أو أعلى.
اقترح لي أفضل 3 خيارات.

OPENAI ANSWER:

إليك أفضل 3 خيارات لفنادق في الرياض لشخصين، بسعر لا يتجاوز 400 ريال وتقييم نزلاء 8 من 10 أو أعلى:

1. **Ewaa Express Hotel - Gaber**
   - السعر: 375 ريال
   - تقييم النزلاء: 9.1
   - تصنيف النجوم: 3 نجوم
   - تاريخ لقطة السعر: 24 أبريل 2020

2. **ضيافة الفيحاء Dayafat Alfayha'a**
   - السعر: 200 ريال
   - تقييم النزلاء: 8.9
   - تصنيف النجوم: 4 نجوم
   - تاريخ لقطة السعر: 24 أبريل 2020

3. **Shaza Riyadh**
   - السعر: 400 ريال
   - تقييم النزلاء: 8.8
   - تصنيف النجوم: 5 نجوم
   - تاريخ لقطة السعر: 24 أبريل 2020

إذا كنت بحاجة إلى مزيد من المعلومات أو المساعدة، فلا تتردد في إخباري!

-----------------------------------------------------------------
SCENARIO S2
----------------------------------